In [13]:
# Your example audio file
file1 = r"Z:\실적(Publication)\국제저널(SCI journal)\2025\MDPI_Symmetry_Sundari\codereview\experiments\vox1_test_wav\wav\id10270\5r0dWxy17C8\00001.wav"

# Change this to another file for verification
# same speaker example: another utterance from id10270
file2 = r"Z:\실적(Publication)\국제저널(SCI journal)\2025\MDPI_Symmetry_Sundari\codereview\experiments\vox1_test_wav\wav\id10270\5r0dWxy17C8\00002.wav"

In [23]:
import os
import torch
import torchaudio
import torch.nn.functional as F
from speechbrain.inference.speaker import EncoderClassifier
from speechbrain.utils.fetching import LocalStrategy

# -----------------------------
# Your dataset paths
# -----------------------------
file1 = r"C:\Users\Sundari\Documents\Dataset\vox1_test_wav\wav\id10270\5r0dWxy17C8\00001.wav"
file2 = r"C:\Users\Sundari\Documents\Dataset\vox1_test_wav\wav\id10270\5r0dWxy17C8\00002.wav"

print("File 1 exists:", os.path.exists(file1))
print("File 2 exists:", os.path.exists(file2))

# -----------------------------
# Device
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# -----------------------------
# Save pretrained model locally
# -----------------------------
model_dir = r"C:\Users\Sundari\Documents\Dataset\pretrained_models\spkrec-ecapa-voxceleb"

classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": device},
    savedir=model_dir,
    local_strategy=LocalStrategy.COPY
)

# -----------------------------
# Audio preprocessing
# -----------------------------
def load_audio_for_ecapa(path, target_sr=16000):
    signal, sr = torchaudio.load(path)

    # Convert to mono
    if signal.shape[0] > 1:
        signal = signal.mean(dim=0, keepdim=True)

    # Resample to 16 kHz
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
        signal = resampler(signal)
        sr = target_sr

    return signal, sr

sig1, sr1 = load_audio_for_ecapa(file1)
sig2, sr2 = load_audio_for_ecapa(file2)

print("Signal 1 shape:", sig1.shape, "SR:", sr1)
print("Signal 2 shape:", sig2.shape, "SR:", sr2)

# -----------------------------
# Extract embeddings
# -----------------------------
with torch.no_grad():
    emb1 = classifier.encode_batch(sig1.to(device))
    emb2 = classifier.encode_batch(sig2.to(device))

print("Embedding 1 shape:", emb1.shape)
print("Embedding 2 shape:", emb2.shape)

# -----------------------------
# Cosine similarity
# -----------------------------
score = F.cosine_similarity(emb1.squeeze(1), emb2.squeeze(1)).item()
print("Cosine score:", score)

# -----------------------------
# Simple decision
# -----------------------------
# You can adjust threshold later
threshold = 0.25
prediction = 1 if score > threshold else 0

print("Prediction:", prediction)
print("Decision:", "Same Speaker" if prediction == 1 else "Different Speaker")

File 1 exists: True
File 2 exists: True
Using device: cpu
Signal 1 shape: torch.Size([1, 133761]) SR: 16000
Signal 2 shape: torch.Size([1, 80001]) SR: 16000
Embedding 1 shape: torch.Size([1, 1, 192])
Embedding 2 shape: torch.Size([1, 1, 192])
Cosine score: 0.7277662754058838
Prediction: 1
Decision: Same Speaker


In [33]:
import os
import torch
import torchaudio
import torch.nn.functional as F
from speechbrain.inference.speaker import EncoderClassifier
from speechbrain.utils.fetching import LocalStrategy

AUDIO_1 = r"C:\Users\Sundari\Documents\Dataset\vox1_test_wav\wav\id10270\5r0dWxy17C8\00001.wav"
AUDIO_2 = r"C:\Users\Sundari\Documents\Dataset\vox1_test_wav\wav\id10271\_pOfzIZVLFQ\00001.wav"
MODEL_DIR = r"C:\Users\Sundari\Documents\Dataset\pretrained_models\speaker_verification_model"
TARGET_SR = 16000
THRESHOLD = 0.25
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_audio(path, target_sr=16000):
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
        waveform = resampler(waveform)
    return waveform

def load_model(model_dir, device):
    return EncoderClassifier.from_hparams(
        source=model_dir,
        savedir=model_dir,
        run_opts={"device": device},
        local_strategy=LocalStrategy.COPY,
    )

def extract_embedding(model, waveform, device):
    with torch.no_grad():
        embedding = model.encode_batch(waveform.to(device))
    return embedding.squeeze(1)

waveform_1 = load_audio(AUDIO_1, TARGET_SR)
waveform_2 = load_audio(AUDIO_2, TARGET_SR)

model = load_model(MODEL_DIR, DEVICE)

embedding_1 = extract_embedding(model, waveform_1, DEVICE)
embedding_2 = extract_embedding(model, waveform_2, DEVICE)

score = F.cosine_similarity(embedding_1, embedding_2).item()
prediction = 1 if score > THRESHOLD else 0

print("Cosine score:", score)
print("Prediction:", prediction)
print("Decision:", "Same Speaker" if prediction == 1 else "Different Speaker")

HFValidationError: Repo id must use alphanumeric chars or '-', '_', '.', '--' and '..' are forbidden, '-' and '.' cannot start or end the name, max length is 96: 'C:\Users\Sundari\Documents\Dataset\pretrained_models\speaker_verification_model'.